# Eliminación Gaussiana y Gauss-Jordan en Python

**Álgebra Lineal, sección A · Universidad Francisco Marroquín**

GRUPO 9:
Valentina Moreno Moreno
Ana Valeria Puertas Ramos
Diego Andrés Escobedo Moran
Josué Rodrigo Agreda Barrios

*Parte 1 (Josué Agreda):* funciones de ambos métodos, ejemplo 4×4 paso a paso, comparación con `np.linalg.solve`, ventaja del pivoteo parcial y limitaciones.
*Parte 2 (Diego Escobedo):* aplicaciones de economía y negocios con más variables (5, 6 y 8 incógnitas).
*Gauss-Jordan:* código de Valeria Puertas.

Contenido:

**Parte 1: métodos y comparación**
1. Librerías y funciones auxiliares
2. Gauss-Jordan (código de Valeria Puertas)
3. Eliminación gaussiana con pivoteo parcial
4. Funciones de comparación
5. Ejemplo 4×4 paso a paso
6. Comparación de los tres métodos
7. Ventaja del pivoteo parcial
8. Sistema grande: exactitud y tiempo
9. Limitaciones
10. Resumen de resultados

**Parte 2: aplicaciones de economía y negocios**

11. Aplicación 1: planificación de producción (5 variables)
12. Aplicación 2: costos de departamentos de servicio, método recíproco (6 variables)
13. Aplicación 3: equilibrio de mercado con bienes relacionados (8 variables)
14. Resumen de las aplicaciones
15. Conclusiones y recomendaciones

## 1. Librerías y funciones auxiliares

`TextoValor` muestra los números como fracciones y `MostrarMatriz` imprime la matriz aumentada `[A | b]`. Las usan las dos funciones de eliminación.

In [1]:
import numpy as np                  #Libreria para trabajar con matrices
from fractions import Fraction      #Para mostrar los numeros como fracciones
import time                         #Para medir cuanto tarda cada metodo

np.set_printoptions(suppress=True)  #Sin notacion cientifica al imprimir

TOL = 1e-10    #Tolerancia: lo mas chico que esto se toma como cero

In [2]:
def TextoValor(valor):

    '''Convierte un numero a texto en forma de fraccion: 0.6667 -> 2/3'''

    #limit_denominator evita fracciones gigantes por el redondeo de la compu
    F = Fraction(valor).limit_denominator(1000)

    #Si el denominador es 1 el numero es entero: se escribe 4 y no 4/1
    if F.denominator == 1:
        return str(F.numerator)

    return f'{F.numerator}/{F.denominator}'      #Si no, se escribe numerador/denominador

In [3]:
def MostrarMatriz(M, num_vars, titulo=''):

    '''Imprime la matriz aumentada [A | b] con la barra separadora'''

    if titulo != '':          #'!=' significa 'distinto de'
        print(titulo)

    #Convierte cada entrada de la matriz a texto (recorre renglones i y columnas j)
    #np.shape(M)[0] son los renglones y np.shape(M)[1] las columnas
    Textos = [[TextoValor(M[i,j]) for j in range(np.shape(M)[1])]
              for i in range(np.shape(M)[0])]

    ancho = max(len(t) for Fila in Textos for t in Fila)   #El texto mas largo: para alinear

    for Fila in Textos:
        #Fila[:num_vars] son los coeficientes y Fila[num_vars:] los terminos independientes
        #El '>' alinea a la derecha usando el ancho calculado arriba
        izq = '  '.join(f'{t:>{ancho}}' for t in Fila[:num_vars])
        der = '  '.join(f'{t:>{ancho}}' for t in Fila[num_vars:])
        print(f'[ {izq}  |  {der} ]')

    print()      #Renglon en blanco para separar un paso del siguiente

## 2. Gauss-Jordan (código de Valeria Puertas)

Se incluye sin cambios para comparar ambos métodos con los mismos datos. `EliminacionGaussJordan` lleva la matriz a su forma escalonada reducida (FERR) y `SolucionSistema` la interpreta: sin solución, solución única o soluciones infinitas.

In [4]:
def EliminacionGaussJordan(A, b, VerPasos='S'):

    '''
    Resuelve un sistema de ecuaciones lineales mediante la
    eliminacion Gauss-Jordan.

    Reduce la matriz aumentada [A | b] a su Forma Escalonada Reducida
    por Renglones (FERR) usando las tres operaciones elementales:
    Ri <-> Rj ,  kRi ,  Ri + kRj

    Despliega paso a paso cada operacion por renglon (opcional)
    '''

    A = np.array(A, dtype=float)            #float obliga a usar decimales al dividir
    b = np.reshape(b,(-1,1)).astype(float)  #El -1 calcula solo los renglones: deja b como columna

    M = np.hstack((A,b))                    #hstack pega b a la derecha de A: matriz aumentada
    (num_ecs, num_vars) = np.shape(A)       #Renglones = ecuaciones, columnas = incognitas

    if VerPasos in ['Y','S','Si','Yes','1']:
        MostrarMatriz(M, num_vars, 'Matriz aumentada [A | b]:')

    fila = 0         #Renglon donde se colocara el siguiente pivote
    Pivotes = []     #Aqui se iran guardando las columnas con entradas principales

    #Se recorren las columnas de izquierda a derecha buscando un pivote en cada una
    for col in range(num_vars):

        if fila >= num_ecs:      #Ya no quedan renglones disponibles
            break

        #a) BUSCAR EL PIVOTE: primer renglon de arriba hacia abajo con entrada no nula
        candidato = None         #None significa 'todavia nada'
        for i in range(fila, num_ecs):          #Solo de 'fila' hacia abajo: arriba ya hay pivotes
            if abs(M[i,col]) > TOL:             #Se compara contra TOL y no contra 0, por el redondeo
                candidato = i
                break

        if candidato == None:    #Columna de puros ceros: esa variable sera libre
            continue

        #b) INTERCAMBIO DE RENGLONES    Ri <-> Rj
        if candidato != fila:
            M[[fila,candidato]] = M[[candidato,fila]]    #Voltea los dos renglones de un solo golpe
            if VerPasos in ['Y','S','Si','Yes','1']:
                #Se suma 1 porque Python cuenta desde 0 pero se escribe 'R1', no 'R0'
                MostrarMatriz(M, num_vars, f'R{fila+1} <-> R{candidato+1}')

        #c) NORMALIZACION: la entrada principal debe quedar en 1    kRi
        pivote = M[fila,col]
        if abs(pivote-1) > TOL:                 #Si ya valia 1 este paso se salta
            M[fila] = M[fila]/pivote            #Divide el renglon completo entre el pivote
            M[np.abs(M) < TOL] = 0.0            #Limpia los residuos de redondeo
            if VerPasos in ['Y','S','Si','Yes','1']:
                #Se muestra 1/pivote porque dividir entre 2 es multiplicar por 1/2
                MostrarMatriz(M, num_vars, f'({TextoValor(1/pivote)}) R{fila+1}')

        #d) ELIMINACION: ceros ARRIBA y ABAJO del pivote    Ri + kRj
        #Recorrer TODOS los renglones es lo que distingue a Gauss-Jordan:
        #la eliminacion gaussiana solo limpiaria hacia abajo
        for i in range(num_ecs):
            if i != fila and abs(M[i,col]) > TOL:    #Ni el renglon del pivote ni los que ya son cero
                factor = M[i,col]                    #El numero que hay que eliminar
                M[i] = M[i] - factor*M[fila]         #Como el pivote vale 1, ahi queda factor - factor = 0
                M[np.abs(M) < TOL] = 0.0
                if VerPasos in ['Y','S','Si','Yes','1']:
                    #Restar un negativo es sumar: asi la etiqueta se lee natural
                    signo = '-' if factor > 0 else '+'
                    MostrarMatriz(M, num_vars,
                                  f'R{i+1} {signo} ({TextoValor(abs(factor))}) R{fila+1}')

        Pivotes.append(col)      #Esta columna si tuvo entrada principal
        fila = fila + 1          #El siguiente pivote va un renglon mas abajo

    return M, Pivotes       #Devuelve la FERR y las columnas con entradas principales

In [5]:
def SolucionSistema(M, Pivotes, Nombres):

    '''
    A partir de la FERR determina si el sistema no tiene solucion,
    tiene solucion unica o tiene soluciones infinitas.
    '''

    num_vars = len(Nombres)      #Cuantas incognitas hay: tambien marca donde empieza la columna de b

    #INCONSISTENCIA: renglon [0 0 ... 0 | c] con c distinto de cero, o sea 0 = c
    for i in range(np.shape(M)[0]):
        #np.all exige que TODOS los coeficientes del renglon sean cero
        if np.all(np.abs(M[i,0:num_vars]) < TOL) and abs(M[i,num_vars]) > TOL:
            print('El sistema NO TIENE SOLUCION (sistema inconsistente).')
            return None

    Libres = [j for j in range(num_vars) if j not in Pivotes]   #Columnas sin entrada principal

    #SOLUCION UNICA: no hay variables libres, cada incognita tiene su pivote
    if Libres == []:
        print('El sistema tiene SOLUCION UNICA\n')
        #enumerate entrega el renglon (i) y la columna (col) al mismo tiempo
        for i, col in enumerate(Pivotes):
            print(f'   {Nombres[col]} = {TextoValor(M[i,num_vars])}')   #El valor esta en la ultima columna
        return np.array([M[i,num_vars] for i in range(len(Pivotes))])

    #SOLUCIONES INFINITAS: a cada variable libre se le asigna un parametro t1, t2, ...
    print('El sistema tiene SOLUCIONES INFINITAS\n')
    print('Variables principales:', ', '.join(Nombres[c] for c in Pivotes))
    print('Variables libres     :', ', '.join(Nombres[c] for c in Libres), '\n')

    Parametros = {col: f't{k+1}' for k, col in enumerate(Libres)}   #Primera libre t1, segunda t2...

    #Cada variable principal se despeja en terminos de los parametros
    for i, col in enumerate(Pivotes):
        texto = TextoValor(M[i,num_vars])        #Empieza con el termino independiente
        for j in Libres:
            coef = M[i,j]
            if abs(coef) > TOL:                  #Las libres con coeficiente cero no se escriben
                #El signo se invierte porque la variable pasa al otro lado al despejar
                signo = '-' if coef > 0 else '+'
                numero = '' if abs(abs(coef)-1) < TOL else TextoValor(abs(coef))   #Se escribe t1, no 1t1
                texto = texto + f' {signo} {numero}{Parametros[j]}'
        print(f'   {Nombres[col]} = {texto}')

    for j in Libres:
        print(f'   {Nombres[j]} = {Parametros[j]}')    #Las libres valen su propio parametro

    return None

## 3. Eliminación gaussiana con pivoteo parcial

**Recibe:** la matriz cuadrada `A`, el vector `b`, los nombres de las incógnitas (opcional) y `VerPasos`.
**Devuelve:** un arreglo con la solución `x`, o `None` si no se puede resolver.

Está organizada en cuatro bloques: (1) preparación de la matriz, (2) selección del pivote con pivoteo parcial, (3) eliminación debajo del pivote, sin convertirlo en 1, y (4) sustitución hacia atrás.

In [6]:
def EliminacionGaussiana(A, b, Nombres=None, VerPasos='N'):

    '''
    Resuelve el sistema Ax = b mediante eliminacion gaussiana con
    pivoteo parcial y sustitucion hacia atras.

    Recibe:   A        -> matriz cuadrada de coeficientes (n x n)
              b        -> terminos independientes (n valores)
              Nombres  -> nombres de las incognitas (opcional)
              VerPasos -> 'S' para mostrar cada operacion por renglon
    Devuelve: arreglo con la solucion x, o None si no se puede resolver
    '''

    #1) PREPARACION DE LA MATRIZ
    A = np.array(A, dtype=float)            #float obliga a usar decimales al dividir
    b = np.reshape(b,(-1,1)).astype(float)  #Deja b como columna
    (num_ecs, n) = np.shape(A)              #Renglones = ecuaciones, columnas = incognitas

    if num_ecs != n:                        #Validacion de dimensiones
        print('ERROR: la matriz A debe ser cuadrada.')
        return None
    if np.shape(b)[0] != n:
        print('ERROR: b debe tener un termino independiente por cada ecuacion.')
        return None

    if Nombres is None:                     #Si no se dan nombres: x1, x2, ..., xn
        Nombres = [f'x{k+1}' for k in range(n)]
    Ver = VerPasos in ['Y','S','Si','Yes','1']

    M = np.hstack((A,b))                    #Matriz aumentada [A | b]
    if Ver:
        MostrarMatriz(M, n, 'Matriz aumentada [A | b]:')

    for col in range(n):                    #Una columna (un pivote) a la vez

        #2) SELECCION DEL PIVOTE: pivoteo parcial
        #Se elige, de 'col' hacia abajo, el renglon con el mayor valor absoluto en la columna
        p = col + np.argmax(np.abs(M[col:, col]))

        if abs(M[p,col]) < TOL:             #Ni el mayor sirve: no hay pivote confiable
            print(f'AVISO: pivote demasiado pequeno en la columna {col+1}.')
            print('El sistema puede no tener solucion, tener infinitas soluciones')
            print('o ser numericamente dificil de resolver.')
            return None

        if p != col:                        #Solo se intercambia cuando hace falta
            M[[col,p]] = M[[p,col]]
            if Ver:
                MostrarMatriz(M, n, f'R{col+1} <-> R{p+1}')
        #3) ELIMINACION: ceros solo DEBAJO del pivote    Ri - (factor)Rcol
        #El pivote NO se convierte en 1: no hace falta para eliminar ni para despejar
        for i in range(col+1, n):
            factor = M[i,col]/M[col,col]
            if abs(factor) > TOL:           #Si ya es cero, no se hace nada
                M[i] = M[i] - factor*M[col]
                M[np.abs(M) < TOL] = 0.0    #Limpia los residuos de redondeo
                if Ver:
                    signo = '-' if factor > 0 else '+'
                    MostrarMatriz(M, n, f'R{i+1} {signo} ({TextoValor(abs(factor))}) R{col+1}')

    #4) SUSTITUCION HACIA ATRAS: del ultimo renglon al primero
    x = np.zeros(n)
    if Ver:
        print('Sustitucion hacia atras:')
    for i in range(n-1, -1, -1):
        suma = np.dot(M[i,i+1:n], x[i+1:n])     #Aporte de las incognitas ya conocidas
        x[i] = (M[i,n] - suma)/M[i,i]           #Se despeja la incognita del renglon i
        if Ver:
            print(f'   {Nombres[i]} = ({TextoValor(M[i,n])} - ({TextoValor(suma)}))'
                  f' / {TextoValor(M[i,i])} = {TextoValor(x[i])}')

    return x

## 4. Funciones de comparación

`CompararConNumpy` compara la eliminación gaussiana con `np.linalg.solve`. `CompararMetodos` compara los tres métodos (Gauss, Gauss-Jordan y NumPy) con los mismos datos: diferencia contra NumPy, residuo `|Ax − b|` y tiempo.

In [7]:
def CompararConNumpy(A, b, Nombres=None):

    '''Resuelve el mismo sistema con nuestra funcion y con np.linalg.solve'''

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    if Nombres is None:
        Nombres = [f'x{k+1}' for k in range(len(b))]

    x_gauss = EliminacionGaussiana(A, b, Nombres)
    x_numpy = np.linalg.solve(A, b)

    print(f'{"Variable":<14}{"Nuestra funcion":>18}{"np.linalg.solve":>18}{"Diferencia":>14}')
    for k in range(len(b)):
        dif = abs(x_gauss[k] - x_numpy[k])
        print(f'{Nombres[k]:<14}{x_gauss[k]:>18.6f}{x_numpy[k]:>18.6f}{dif:>14.1e}')

    #Residuo: que tan lejos queda A x de b (0 seria perfecto)
    print('\nDiferencia maxima entre soluciones:', f'{np.max(np.abs(x_gauss - x_numpy)):.1e}')
    print('Residuo maximo |Ax - b| (nuestra):  ', f'{np.max(np.abs(A @ x_gauss - b)):.1e}')
    print('Residuo maximo |Ax - b| (numpy):    ', f'{np.max(np.abs(A @ x_numpy - b)):.1e}')

In [8]:
def CompararMetodos(A, b, Nombres=None, Mostrar='S'):

    '''
    Resuelve el mismo sistema con eliminacion gaussiana, Gauss-Jordan
    y np.linalg.solve, y compara exactitud y tiempo.

    Recibe:   A, b, Nombres (opcional) y Mostrar ('S' imprime las tablas)
    Devuelve: diccionario {metodo: solucion} con los metodos que si resolvieron
    '''

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    if Nombres is None:
        Nombres = [f'x{k+1}' for k in range(n)]

    Soluciones = {}      #Solucion de cada metodo
    Tiempos = {}         #Tiempo de cada metodo en segundos

    #Se mide el tiempo de cada metodo con time.perf_counter()
    inicio = time.perf_counter()
    x = EliminacionGaussiana(A, b, Nombres)
    Tiempos['Gauss'] = time.perf_counter() - inicio
    if x is not None:
        Soluciones['Gauss'] = x

    inicio = time.perf_counter()
    FERR, Pivotes = EliminacionGaussJordan(A, b, VerPasos='N')
    Tiempos['Gauss-Jordan'] = time.perf_counter() - inicio
    if Pivotes == list(range(n)):           #Un pivote en cada columna: solucion unica
        Soluciones['Gauss-Jordan'] = FERR[:n, n]    #En la FERR la solucion esta en la ultima columna
    else:
        print('Gauss-Jordan no encontro un pivote en cada columna.')

    try:                                    #NumPy detiene el programa si la matriz es singular
        inicio = time.perf_counter()
        Soluciones['NumPy'] = np.linalg.solve(A, b)
        Tiempos['NumPy'] = time.perf_counter() - inicio
    except np.linalg.LinAlgError as error:
        print('np.linalg.solve no pudo resolver:', error)

    if Mostrar in ['Y','S','Si','Yes','1'] and len(Soluciones) > 0:
        Metodos = list(Soluciones)
        if n <= 12:                         #La tabla por variable solo si el sistema es pequeno
            print(f'{"Variable":<14}' + ''.join(f'{m:>14}' for m in Metodos))
            for k in range(n):
                print(f'{Nombres[k]:<14}' + ''.join(f'{Soluciones[m][k]:>14.6f}' for m in Metodos))
            print()

        print(f'{"Metodo":<14}{"Dif. vs NumPy":>15}{"Residuo |Ax-b|":>17}{"Tiempo (ms)":>14}')
        for m in Metodos:
            if 'NumPy' in Soluciones:
                dif = f'{np.max(np.abs(Soluciones[m] - Soluciones["NumPy"])):.1e}'
            else:
                dif = '-'
            residuo = np.max(np.abs(A @ Soluciones[m] - b))      #Que tan lejos queda Ax de b
            print(f'{m:<14}{dif:>15}{residuo:>17.1e}{Tiempos[m]*1000:>14.3f}')

    return Soluciones

## 5. Ejemplo 4×4 paso a paso

Se usa el mismo sistema de la sección de Gauss-Jordan del reporte.

In [9]:
#        2x +  y -  z =  2
#   w +   x -  y + 2z =  6
#  2w -   x + 3y +  z = -2
#  3w +  2x +  y -  z =  5

A = [[0,  2,  1, -1],
     [1,  1, -1,  2],
     [2, -1,  3,  1],
     [3,  2,  1, -1]]

b = [2, 6, -2, 5]

Nombres = ['w', 'x', 'y', 'z']

### 5.1 Eliminación gaussiana

Se muestran los intercambios del pivoteo parcial y cómo se eliminan las entradas debajo de cada pivote.

In [10]:
Solucion = EliminacionGaussiana(A, b, Nombres, VerPasos='S')
print('Solucion:', [TextoValor(v) for v in Solucion])

Matriz aumentada [A | b]:
[  0   2   1  -1  |   2 ]
[  1   1  -1   2  |   6 ]
[  2  -1   3   1  |  -2 ]
[  3   2   1  -1  |   5 ]

R1 <-> R4
[  3   2   1  -1  |   5 ]
[  1   1  -1   2  |   6 ]
[  2  -1   3   1  |  -2 ]
[  0   2   1  -1  |   2 ]

R2 - (1/3) R1
[    3     2     1    -1  |     5 ]
[    0   1/3  -4/3   7/3  |  13/3 ]
[    2    -1     3     1  |    -2 ]
[    0     2     1    -1  |     2 ]

R3 - (2/3) R1
[     3      2      1     -1  |      5 ]
[     0    1/3   -4/3    7/3  |   13/3 ]
[     0   -7/3    7/3    5/3  |  -16/3 ]
[     0      2      1     -1  |      2 ]

R2 <-> R3
[     3      2      1     -1  |      5 ]
[     0   -7/3    7/3    5/3  |  -16/3 ]
[     0    1/3   -4/3    7/3  |   13/3 ]
[     0      2      1     -1  |      2 ]

R3 + (1/7) R2
[     3      2      1     -1  |      5 ]
[     0   -7/3    7/3    5/3  |  -16/3 ]
[     0      0     -1   18/7  |   25/7 ]
[     0      2      1     -1  |      2 ]

R4 + (6/7) R2
[     3      2      1     -1  |      5 ]
[     0

### 5.2 Gauss-Jordan

Este método normaliza cada pivote a 1 y elimina arriba y abajo de él.

In [11]:
FERR, Pivotes = EliminacionGaussJordan(A, b, VerPasos='S')
SolucionSistema(FERR, Pivotes, Nombres)

Matriz aumentada [A | b]:
[  0   2   1  -1  |   2 ]
[  1   1  -1   2  |   6 ]
[  2  -1   3   1  |  -2 ]
[  3   2   1  -1  |   5 ]

R1 <-> R2
[  1   1  -1   2  |   6 ]
[  0   2   1  -1  |   2 ]
[  2  -1   3   1  |  -2 ]
[  3   2   1  -1  |   5 ]

R3 - (2) R1
[   1    1   -1    2  |    6 ]
[   0    2    1   -1  |    2 ]
[   0   -3    5   -3  |  -14 ]
[   3    2    1   -1  |    5 ]

R4 - (3) R1
[   1    1   -1    2  |    6 ]
[   0    2    1   -1  |    2 ]
[   0   -3    5   -3  |  -14 ]
[   0   -1    4   -7  |  -13 ]

(1/2) R2
[    1     1    -1     2  |     6 ]
[    0     1   1/2  -1/2  |     1 ]
[    0    -3     5    -3  |   -14 ]
[    0    -1     4    -7  |   -13 ]

R1 - (1) R2
[    1     0  -3/2   5/2  |     5 ]
[    0     1   1/2  -1/2  |     1 ]
[    0    -3     5    -3  |   -14 ]
[    0    -1     4    -7  |   -13 ]

R3 + (3) R2
[    1     0  -3/2   5/2  |     5 ]
[    0     1   1/2  -1/2  |     1 ]
[    0     0  13/2  -9/2  |   -11 ]
[    0    -1     4    -7  |   -13 ]

R4 + (1) R2


array([ 1.,  2., -1.,  1.])

## 6. Comparación de los tres métodos

Mismo sistema 4×4 resuelto con Gauss, Gauss-Jordan y `np.linalg.solve`.

In [12]:
Resultados4x4 = CompararMetodos(A, b, Nombres)

Variable               Gauss  Gauss-Jordan         NumPy
w                   1.000000      1.000000      1.000000
x                   2.000000      2.000000      2.000000
y                  -1.000000     -1.000000     -1.000000
z                   1.000000      1.000000      1.000000

Metodo          Dif. vs NumPy   Residuo |Ax-b|   Tiempo (ms)
Gauss                 2.2e-16          8.9e-16         0.391
Gauss-Jordan          4.4e-16          0.0e+00         0.097
NumPy                 0.0e+00          8.9e-16         0.135


Comparación detallada de la eliminación gaussiana con NumPy (la que aparece en el reporte):

In [13]:
CompararConNumpy(A, b, Nombres)

Variable         Nuestra funcion   np.linalg.solve    Diferencia
w                       1.000000          1.000000       0.0e+00
x                       2.000000          2.000000       0.0e+00
y                      -1.000000         -1.000000       0.0e+00
z                       1.000000          1.000000       2.2e-16

Diferencia maxima entre soluciones: 2.2e-16
Residuo maximo |Ax - b| (nuestra):   8.9e-16
Residuo maximo |Ax - b| (numpy):     8.9e-16


## 7. Ventaja del pivoteo parcial

Sistema εx + y = 1, x + y = 2 con ε = 10⁻⁹. Su solución exacta es x = 1/(1 − ε) ≈ 1.000000001 e y = (1 − 2ε)/(1 − ε) ≈ 0.999999999.

Gauss-Jordan (sin pivoteo parcial) usa ε como pivote porque es distinto de cero. La eliminación gaussiana con pivoteo parcial intercambia los renglones y usa el 1 como pivote.

In [14]:
eps = 1e-9
A_eps = [[eps, 1],
         [1,   1]]
b_eps = [1, 2]

x_exacta = np.array([1/(1-eps), (1-2*eps)/(1-eps)])      #Solucion calculada a mano

Soluciones = CompararMetodos(A_eps, b_eps, ['x', 'y'], Mostrar='N')
print(f'{"Metodo":<14}{"x":>22}{"Error en x":>14}')
for metodo, x in Soluciones.items():
    print(f'{metodo:<14}{x[0]:>22.15f}{abs(x[0]-x_exacta[0]):>14.1e}')

Metodo                             x    Error en x
Gauss              1.000000001000000       0.0e+00
Gauss-Jordan       1.000000000000000       1.0e-09
NumPy              1.000000001000000       0.0e+00


## 8. Sistema grande: exactitud y tiempo

Sistema 100×100 con coeficientes aleatorios. La semilla fija hace que siempre salgan los mismos números. Los tiempos dependen de la computadora.

In [15]:
np.random.seed(1)                   #Semilla: siempre los mismos numeros aleatorios
A_grande = np.random.rand(100, 100)
b_grande = np.random.rand(100)

ResultadosGrande = CompararMetodos(A_grande, b_grande)

Metodo          Dif. vs NumPy   Residuo |Ax-b|   Tiempo (ms)
Gauss                 2.7e-14          1.8e-14        47.328
Gauss-Jordan          2.0e-12          1.1e-12        99.998
NumPy                 0.0e+00          1.5e-14         0.311


## 9. Limitaciones

### 9.1 Sistema rectangular (2 ecuaciones, 3 incógnitas)

La eliminación gaussiana solo admite matrices cuadradas. Gauss-Jordan sí lo resuelve y expresa la solución con parámetros.

In [16]:
A_rect = [[1, 2, 3],
          [2, 4, 1]]
b_rect = [1, 2]

print('Eliminacion gaussiana:')
EliminacionGaussiana(A_rect, b_rect)

print('\nGauss-Jordan:')
FERR, Pivotes = EliminacionGaussJordan(A_rect, b_rect, VerPasos='N')
SolucionSistema(FERR, Pivotes, ['x', 'y', 'z'])

Eliminacion gaussiana:
ERROR: la matriz A debe ser cuadrada.

Gauss-Jordan:
El sistema tiene SOLUCIONES INFINITAS

Variables principales: x, z
Variables libres     : y 

   x = 1 - 2t1
   z = 0
   y = t1


### 9.2 Infinitas soluciones contra sin solución

Los dos sistemas tienen la misma matriz A y solo cambia b. La eliminación gaussiana da el mismo aviso en ambos casos; Gauss-Jordan sí los distingue.

In [17]:
A_sing = [[1, 2],
          [2, 4]]

for b_sing in ([3, 6], [3, 7]):
    print(f'b = {b_sing}')
    print('Eliminacion gaussiana:')
    EliminacionGaussiana(A_sing, b_sing)
    print('Gauss-Jordan:')
    FERR, Pivotes = EliminacionGaussJordan(A_sing, b_sing, VerPasos='N')
    SolucionSistema(FERR, Pivotes, ['x', 'y'])
    print()

b = [3, 6]
Eliminacion gaussiana:
AVISO: pivote demasiado pequeno en la columna 2.
El sistema puede no tener solucion, tener infinitas soluciones
o ser numericamente dificil de resolver.
Gauss-Jordan:
El sistema tiene SOLUCIONES INFINITAS

Variables principales: x
Variables libres     : y 

   x = 3 - 2t1
   y = t1

b = [3, 7]
Eliminacion gaussiana:
AVISO: pivote demasiado pequeno en la columna 2.
El sistema puede no tener solucion, tener infinitas soluciones
o ser numericamente dificil de resolver.
Gauss-Jordan:
El sistema NO TIENE SOLUCION (sistema inconsistente).



### 9.3 Sistemas numéricamente difíciles (matriz de Hilbert)

La matriz de Hilbert, con entradas 1/(i + j + 1), es invertible pero está muy mal condicionada. Se construye b para que la solución exacta sea x = (1, 1, …, 1) y se mide el error de cada método. El número de condición indica qué tan sensible es el sistema a los errores de redondeo.

In [18]:
print(f'{"n":>3}{"Condicion":>12}{"Error Gauss":>14}{"Error G-J":>12}{"Error NumPy":>14}')
for n in [6, 8, 10, 12]:
    H = np.array([[1/(i+j+1) for j in range(n)] for i in range(n)])
    b_H = H @ np.ones(n)                    #Asi la solucion exacta es puros unos

    Soluciones = CompararMetodos(H, b_H, Mostrar='N')
    Errores = [f'{np.max(np.abs(Soluciones[m] - 1)):.1e}' if m in Soluciones else 'fallo'
               for m in ['Gauss', 'Gauss-Jordan', 'NumPy']]
    print(f'{n:>3}{np.linalg.cond(H):>12.1e}{Errores[0]:>14}{Errores[1]:>12}{Errores[2]:>14}')

  n   Condicion   Error Gauss   Error G-J   Error NumPy
  6     1.5e+07       3.2e-10     3.9e-10       6.6e-11
  8     1.5e+10       7.0e-08     5.1e-08       1.3e-07
Gauss-Jordan no encontro un pivote en cada columna.
 10     1.6e+13       2.0e+00       fallo       2.4e-05
AVISO: pivote demasiado pequeno en la columna 10.
El sistema puede no tener solucion, tener infinitas soluciones
o ser numericamente dificil de resolver.
Gauss-Jordan no encontro un pivote en cada columna.
 12     1.6e+16         fallo       fallo       2.8e-01


## 10. Resumen de resultados

| Prueba | Eliminación gaussiana | Gauss-Jordan | `np.linalg.solve` |
|---|---|---|---|
| Ejemplo 4×4 | (1, 2, −1, 1); diferencia con NumPy 2.2×10⁻¹⁶ | (1, 2, −1, 1); diferencia 4.4×10⁻¹⁶ | (1, 2, −1, 1) |
| Pivote pequeño (ε = 10⁻⁹) | Error 0 (pivoteo parcial) | Error 1.0×10⁻⁹ (usa ε como pivote) | Error 0 |
| Aleatorio 100×100 | Diferencia 2.7×10⁻¹⁴; residuo 1.8×10⁻¹⁴ | Diferencia 2.0×10⁻¹²; residuo 1.1×10⁻¹² | Residuo 1.5×10⁻¹⁴ |
| Tiempo 100×100 (esta ejecución) | ≈ 47 ms | ≈ 100 ms | ≈ 0.3 ms |
| Rectangular 2×3 | Error: A debe ser cuadrada | Infinitas soluciones con parámetro | No aplica (solo cuadradas) |
| Singular, infinitas soluciones | Aviso de pivote pequeño | Infinitas soluciones | `LinAlgError` |
| Singular, sin solución | El mismo aviso | Sin solución | `LinAlgError` |
| Hilbert 10×10 | Error 2.0 sin aviso | No encuentra pivote | Error 2.4×10⁻⁵ |

**Observaciones**

- En sistemas con solución única, los tres métodos coinciden hasta errores de redondeo.
- El pivoteo parcial hace a la eliminación gaussiana más exacta que el Gauss-Jordan del proyecto (que toma el primer pivote distinto de cero), como se ve con ε y en el sistema 100×100.
- `np.linalg.solve` es cientos de veces más rápido porque usa LAPACK, que está compilado. Internamente también usa factorización LU con pivoteo parcial.
- Gauss-Jordan es el único de los tres que clasifica sistemas sin solución o con infinitas soluciones.
- La tolerancia fija `TOL = 1e-10` es absoluta. En la matriz de Hilbert 10×10 el pivote más pequeño es del orden de 10⁻¹², menor que `TOL`. La limpieza de residuos borra esos valores y nuestras funciones fallan, mientras que NumPy, que no usa una tolerancia fija, conserva cinco cifras correctas.

# Parte 2: aplicaciones de economía y negocios

Se usan las mismas funciones definidas en las secciones 1 a 4. Cada aplicación se plantea como un sistema Ax = b y se resuelve con `CompararMetodos(A, b, Nombres)`, que usa la eliminación gaussiana, Gauss-Jordan y `np.linalg.solve`, e imprime la diferencia contra NumPy, el residuo |Ax − b| y el tiempo de cada método.

**Aviso:** los datos de las tres aplicaciones son **ilustrativos** (inventados para el ejemplo), no datos reales.

## 11. Aplicación 1: planificación de producción (5 variables)

**Contexto.** Una mueblería en Guatemala fabrica cinco productos. Cada semana tiene una cantidad fija de horas de máquina, de mano de obra y de materiales. La gerencia quiere saber cuántas unidades de cada producto fabricar para **usar exactamente** todos los recursos disponibles, sin que sobre ni falte nada.

**Variables** (unidades producidas por semana):

| Variable | Producto |
|---|---|
| x₁ | Sillas |
| x₂ | Mesas |
| x₃ | Libreras |
| x₄ | Escritorios |
| x₅ | Camas |

**De dónde salen las ecuaciones.** Hay una ecuación por recurso: lo que gasta cada producto por unidad, multiplicado por las unidades producidas, sumado para los cinco productos, debe ser igual a lo disponible. Por ejemplo, para las horas de corte:

$$1x_1 + 3x_2 + 2x_3 + 4x_4 + 5x_5 = 193$$

Cada columna de A es un producto y cada renglón es un recurso.

In [19]:
#Cada renglon es un recurso y cada columna un producto
#Columnas:   Sillas  Mesas  Libreras  Escritorios  Camas
A1 = [[  1,     3,     2,       4,         5  ],    #Corte (horas de maquina por unidad)
      [  2,     2,     3,       3,         4  ],    #Ensamble (horas de trabajo por unidad)
      [  1,     2,     2,       3,         3  ],    #Acabado (horas por unidad)
      [  8,    25,    30,      35,        45  ],    #Madera (pies tablares por unidad)
      [0.5,   1.5,     2,     1.5,       2.5  ]]    #Barniz (litros por unidad)

#Recursos disponibles en la semana (datos ilustrativos)
b1 = [193,      #Horas de corte
      208,      #Horas de ensamble
      150,      #Horas de acabado
      1775,     #Pies tablares de madera
      100.5]    #Litros de barniz

Nombres1 = ['Sillas', 'Mesas', 'Libreras', 'Escritorios', 'Camas']

In [20]:
Resultados1 = CompararMetodos(A1, b1, Nombres1)     #Resuelve con los tres metodos y compara

Variable               Gauss  Gauss-Jordan         NumPy
Sillas             40.000000     40.000000     40.000000
Mesas              15.000000     15.000000     15.000000
Libreras           10.000000     10.000000     10.000000
Escritorios        12.000000     12.000000     12.000000
Camas               8.000000      8.000000      8.000000

Metodo          Dif. vs NumPy   Residuo |Ax-b|   Tiempo (ms)
Gauss                 1.2e-14          2.8e-14         0.423
Gauss-Jordan          2.1e-14          0.0e+00         0.218
NumPy                 0.0e+00          0.0e+00         0.062


In [21]:
x1 = Resultados1['Gauss']      #Se usa la solucion de nuestra funcion

#Precio de venta de cada producto en quetzales (ilustrativo)
Precios = np.array([350, 1200, 1500, 1800, 2500])

print('Plan de produccion semanal:')
for k in range(5):
    print(f'   {Nombres1[k]:<12} {x1[k]:6.1f} unidades')

print(f'\nIngreso por ventas: Q{Precios @ x1:,.2f}')     #@ multiplica y suma: precio x cantidad

#Verificacion: no puede haber produccion negativa
print('\nTodas las cantidades son positivas:', np.all(x1 >= 0))
print('Recursos usados (A x):', A1 @ x1)                 #Debe dar exactamente b1

Plan de produccion semanal:
   Sillas         40.0 unidades
   Mesas          15.0 unidades
   Libreras       10.0 unidades
   Escritorios    12.0 unidades
   Camas           8.0 unidades

Ingreso por ventas: Q88,600.00

Todas las cantidades son positivas: True
Recursos usados (A x): [ 193.   208.   150.  1775.   100.5]


**Interpretación.** Para usar exactamente todos los recursos de la semana, la mueblería debe fabricar **40 sillas, 15 mesas, 10 libreras, 12 escritorios y 8 camas**. Con los precios del ejemplo, eso deja un ingreso de **Q88,600 a la semana**.

Como el sistema tiene solución única, **solo una combinación de productos** usa exactamente las horas, la madera y el barniz disponibles. Si algún resultado hubiera salido negativo, querría decir que con esos recursos no se puede producir sin que algo sobre o falte, y habría que cambiar el plan.

**Verificación:** las cinco cantidades son positivas y al multiplicar A·x se obtienen exactamente los recursos disponibles (193, 208, 150, 1775 y 100.5). Los tres métodos dan la misma solución.

## 12. Aplicación 2: costos de departamentos de servicio, método recíproco (6 variables)

**Contexto.** Una empresa tiene seis departamentos de servicio que no venden nada, pero se prestan servicios entre sí y a los dos departamentos que sí generan ingresos (Producción y Ventas). Por ejemplo, TI le da soporte a Mantenimiento y Mantenimiento le repara equipo a TI. En contabilidad de costos, el **método recíproco** reparte el costo de los departamentos de servicio tomando en cuenta estos servicios cruzados. Así se sabe cuánto cuesta en realidad cada departamento operativo.

**Variables** (costo total de cada departamento de servicio, en miles de quetzales al mes):

| Variable | Departamento |
|---|---|
| T₁ | Mantenimiento |
| T₂ | TI |
| T₃ | Recursos Humanos |
| T₄ | Limpieza |
| T₅ | Contabilidad |
| T₆ | Seguridad |

**De dónde salen las ecuaciones.** El costo total de cada departamento es su costo directo más la parte que recibe de los servicios de los demás:

$$T_i = D_i + \sum_j p_{ij}\,T_j$$

Aquí $p_{ij}$ es la fracción de los servicios del departamento $j$ que usa el departamento $i$. Al pasar todo a la izquierda queda el sistema

$$(I - P)\,T = D$$

In [22]:
Nombres2 = ['Mantenimiento', 'TI', 'RRHH', 'Limpieza', 'Contabilidad', 'Seguridad']

#P[i][j] = fraccion de los servicios del departamento j que usa el departamento i
#Columnas:  Mant   TI    RRHH  Limp  Cont  Seg
P = np.array([[0,    0.10, 0.05, 0.10, 0.05, 0.10],    #Mantenimiento
              [0.15, 0,    0.10, 0.05, 0.15, 0.10],    #TI
              [0.05, 0.10, 0,    0.10, 0.10, 0.05],    #RRHH
              [0.10, 0.05, 0.05, 0,    0.05, 0.05],    #Limpieza
              [0.05, 0.10, 0.15, 0.05, 0,    0.10],    #Contabilidad
              [0.10, 0.05, 0.05, 0.10, 0.05, 0   ]])   #Seguridad

#Costos directos de cada departamento en miles de quetzales al mes (ilustrativos)
D = [120, 150, 90, 60, 80, 70]

A2 = np.eye(6) - P     #np.eye(6) es la matriz identidad de 6 x 6

In [23]:
Resultados2 = CompararMetodos(A2, D, Nombres2)

Variable               Gauss  Gauss-Jordan         NumPy
Mantenimiento     182.763854    182.763854    182.763854
TI                234.344060    234.344060    234.344060
RRHH              155.501795    155.501795    155.501795
Limpieza          111.809246    111.809246    111.809246
Contabilidad      154.154061    154.154061    154.154061
Seguridad         126.657306    126.657306    126.657306

Metodo          Dif. vs NumPy   Residuo |Ax-b|   Tiempo (ms)
Gauss                 2.8e-14          4.3e-14         0.596
Gauss-Jordan          2.8e-14          1.4e-14         0.153
NumPy                 0.0e+00          4.3e-14         0.046


**Reparto a los departamentos operativos.** El resto de los servicios de cada departamento (lo que no usan los demás departamentos de servicio) se va a Producción y a Ventas. Con los costos totales T se calcula cuánto le toca a cada uno.

In [24]:
T = Resultados2['Gauss']

#Fraccion de los servicios de cada departamento que usan Produccion y Ventas
#Columnas:          Mant  TI    RRHH  Limp  Cont  Seg
Reparto = np.array([[0.35, 0.40, 0.30, 0.40, 0.30, 0.35],    #Produccion
                    [0.20, 0.20, 0.30, 0.20, 0.30, 0.25]])   #Ventas

#Revision de datos: cada columna de P y Reparto debe sumar 1 (100% del servicio)
print('Cada departamento reparte el 100% de su servicio:', np.allclose(P.sum(axis=0) + Reparto.sum(axis=0), 1))

print('\nCosto total de cada departamento de servicio (miles de Q):')
for k in range(6):
    print(f'   {Nombres2[k]:<14} {T[k]:8.2f}   (directo: {D[k]})')

Asignado = Reparto @ T     #Costo que recibe cada departamento operativo
print(f'\nCosto asignado a Produccion: Q{Asignado[0]:.2f} miles')
print(f'Costo asignado a Ventas:     Q{Asignado[1]:.2f} miles')

#Verificacion: lo asignado debe ser igual a la suma de los costos directos
print(f'\nSuma de costos directos: {sum(D)}   Total asignado: {Asignado.sum():.2f}')

Cada departamento reparte el 100% de su servicio: True

Costo total de cada departamento de servicio (miles de Q):
   Mantenimiento    182.76   (directo: 120)
   TI               234.34   (directo: 150)
   RRHH             155.50   (directo: 90)
   Limpieza         111.81   (directo: 60)
   Contabilidad     154.15   (directo: 80)
   Seguridad        126.66   (directo: 70)

Costo asignado a Produccion: Q339.66 miles
Costo asignado a Ventas:     Q230.34 miles

Suma de costos directos: 570   Total asignado: 570.00


**Interpretación.** El costo real de cada departamento de servicio es **mayor que su costo directo**, porque también incluye los servicios que recibe de los demás. Por ejemplo, Mantenimiento gasta Q120 mil directamente, pero su costo total es Q182.76 mil. TI es el más caro (Q234.34 mil), porque tiene el mayor costo directo y además es el que más servicios recibe de los otros departamentos.

Al repartir esos costos, a **Producción le tocan Q339.66 mil** y a **Ventas, Q230.34 mil** al mes. Con esto la empresa puede fijar precios o evaluar cada área con su costo completo y no solo con el directo.

**Verificación:** el total asignado a Producción y Ventas (Q570 mil) es igual a la suma de los costos directos. El método no pierde ni duplica costos, aunque la suma de los costos totales T sea mayor, porque ahí los servicios cruzados se cuentan en los dos departamentos. Además, cada departamento reparte exactamente el 100 % de su servicio.

## 13. Aplicación 3: equilibrio de mercado con bienes relacionados (8 variables)

**Contexto.** En un mercado de la canasta básica, el precio de un producto también afecta la demanda de otros. Si sube el café se compra menos azúcar, porque son **complementarios**. Si sube el pan se compran más tortillas, porque son **sustitutos**. Se busca el **precio de equilibrio** de los ocho productos, es decir, los precios a los que la cantidad ofrecida es igual a la demandada en todos los mercados a la vez.

**Variables:** el precio de cada producto en quetzales por unidad de venta: café, azúcar, leche, pan, tortillas, frijol, arroz y huevos.

**De dónde salen las ecuaciones.** Para cada producto $i$ (cantidades en miles de unidades a la semana):

- **Demanda:** $Q^d_i = a_i + \sum_j e_{ij}\,p_j$. Si $e_{ii} < 0$, la demanda baja cuando sube el propio precio. Si $e_{ij} < 0$, los bienes son complementarios, y si $e_{ij} > 0$, son sustitutos.
- **Oferta:** $Q^s_i = c_i + s_i\,p_i$. Los productores ofrecen más cuando sube el precio.

En el equilibrio $Q^s_i = Q^d_i$, y al pasar los precios a la izquierda queda

$$s_i\,p_i - \sum_j e_{ij}\,p_j = a_i - c_i \quad\Longrightarrow\quad (\text{diag}(s) - E)\,p = a - c$$

In [25]:
Nombres3 = ['Cafe', 'Azucar', 'Leche', 'Pan', 'Tortillas', 'Frijol', 'Arroz', 'Huevos']

#E[i][j] = cambio en la demanda del bien i cuando el precio del bien j sube Q1
#Negativo en la diagonal (precio propio), negativo = complementarios, positivo = sustitutos
#Columnas:   Cafe  Azuc  Lech  Pan   Tort  Frij  Arro  Huev
E = np.array([[-3.0, -0.5, -0.4,  0,    0,    0,    0,    0  ],    #Cafe
              [-0.6, -4.0,  0,   -0.3,  0,    0,    0,    0  ],    #Azucar
              [-0.5,  0,   -3.5, -0.4,  0,    0,    0,    0  ],    #Leche
              [ 0,   -0.3, -0.4, -5.0,  0.8,  0,    0,   -0.3],    #Pan
              [ 0,    0,    0,    0.6, -6.0, -0.5,  0.4,  0  ],    #Tortillas
              [ 0,    0,    0,    0,   -0.7, -4.0, -0.6, -0.2],    #Frijol
              [ 0,    0,    0,    0,    0.5, -0.5, -4.5,  0  ],    #Arroz
              [ 0,    0,    0,   -0.3,  0,   -0.2,  0,   -3.0]])   #Huevos

a = np.array([150, 120, 140, 110, 160, 130, 120, 100])   #Demanda si todos los precios fueran 0
c = np.array([ 20,  10,  15,   5,  10,  10,   8,   6])   #Oferta si el precio fuera 0
s = np.array([2.0, 3.0, 2.5, 4.0, 5.0, 3.0, 3.5, 2.5])   #Aumento de la oferta por cada Q1 de precio

A3 = np.diag(s) - E     #np.diag pone s en la diagonal de una matriz de ceros
b3 = a - c

In [26]:
Resultados3 = CompararMetodos(A3, b3, Nombres3)

Variable               Gauss  Gauss-Jordan         NumPy
Cafe               23.222938     23.222938     23.222938
Azucar             13.246304     13.246304     13.246304
Leche              18.155398     18.155398     18.155398
Pan                11.140362     11.140362     11.140362
Tortillas          14.113426     14.113426     14.113426
Frijol             14.074982     14.074982     14.074982
Arroz              14.002403     14.002403     14.002403
Huevos             15.971435     15.971435     15.971435

Metodo          Dif. vs NumPy   Residuo |Ax-b|   Tiempo (ms)
Gauss                 1.8e-15          2.8e-14         0.316
Gauss-Jordan          3.6e-15          2.8e-14         0.180
NumPy                 0.0e+00          2.8e-14         0.042


In [27]:
Precios3 = Resultados3['Gauss']
Cantidades = c + s*Precios3      #Cantidad de equilibrio (oferta = demanda)

print(f'{"Producto":<12}{"Precio (Q)":>12}{"Cantidad (miles)":>18}')
for k in range(8):
    print(f'{Nombres3[k]:<12}{Precios3[k]:>12.2f}{Cantidades[k]:>18.2f}')

#Verificacion: precios y cantidades positivos, y oferta igual a demanda
Demanda = a + E @ Precios3
print('\nPrecios y cantidades positivos:', np.all(Precios3 > 0) and np.all(Cantidades > 0))
print('Oferta = demanda en los 8 mercados:', np.allclose(Cantidades, Demanda))

Producto      Precio (Q)  Cantidad (miles)
Cafe               23.22             66.45
Azucar             13.25             49.74
Leche              18.16             60.39
Pan                11.14             49.56
Tortillas          14.11             80.57
Frijol             14.07             52.22
Arroz              14.00             57.01
Huevos             15.97             45.93

Precios y cantidades positivos: True
Oferta = demanda en los 8 mercados: True


### 13.1 Escenario: sube el costo de producir café

Una plaga (por ejemplo, la roya) reduce la oferta de café: con el mismo precio, los productores ahora ofrecen 15 mil unidades menos (c₁ pasa de 20 a 5). Solo cambia el vector b y se resuelve de nuevo. Así se ve cómo el choque en un solo mercado afecta los precios de los demás.

In [28]:
c_nuevo = c.copy()
c_nuevo[0] = 5              #Menos oferta de cafe

Nuevos = CompararMetodos(A3, a - c_nuevo, Nombres3, Mostrar='N')['Gauss']   #Solo cambia b

print(f'{"Producto":<12}{"Antes (Q)":>11}{"Despues (Q)":>13}{"Cambio":>9}')
for k in range(8):
    print(f'{Nombres3[k]:<12}{Precios3[k]:>11.2f}{Nuevos[k]:>13.2f}{Nuevos[k]-Precios3[k]:>+9.2f}')

Producto      Antes (Q)  Despues (Q)   Cambio
Cafe              23.22        26.27    +3.05
Azucar            13.25        12.98    -0.26
Leche             18.16        17.90    -0.26
Pan               11.14        11.16    +0.02
Tortillas         14.11        14.11    +0.00
Frijol            14.07        14.07    -0.00
Arroz             14.00        14.00    +0.00
Huevos            15.97        15.97    -0.00


**Interpretación.** En el equilibrio, el café es el producto más caro (Q23.22) y las tortillas son las que más se venden (80.57 mil unidades). En los ocho mercados la cantidad ofrecida es igual a la demandada.

**Escenario.** Cuando baja la oferta de café, su precio sube **Q3.05 (alrededor de 13 %)**. Como el café es complementario del azúcar y la leche, la gente compra menos de ambos y sus precios **bajan Q0.26 cada uno**. El pan cambia muy poco (+Q0.02), porque solo se relaciona con el café a través del azúcar y la leche. Las tortillas, el frijol, el arroz y los huevos prácticamente no cambian, porque no están relacionados con el café. Esto se ve solo porque se resuelven los ocho mercados al mismo tiempo: analizar el café por separado no muestra el efecto sobre los demás productos.

Para el escenario **solo cambió el vector b**; la matriz A es la misma.

**Verificación:** los precios y las cantidades son positivos, y la oferta es igual a la demanda en los ocho mercados.

## 14. Resumen de las aplicaciones

Las tres aplicaciones, de 5, 6 y 8 variables, se resolvieron con los mismos tres métodos. El número de condición (`np.linalg.cond`) mide qué tan sensible es cada sistema a los errores de redondeo; en la sección 9.3, la matriz de Hilbert 10×10 tenía 1.6×10¹³.

In [29]:
Aplicaciones = [('Produccion', A1, b1), ('Costos', A2, D), ('Mercado', A3, b3)]

print(f'{"Aplicacion":<12}{"Tamano":>8}{"Condicion":>11}{"Dif. Gauss":>12}{"Dif. G-J":>11}{"Res. Gauss":>12}{"Res. G-J":>11}{"Res. NumPy":>12}')
for nombre, A, b in Aplicaciones:
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    R = CompararMetodos(A, b, Mostrar='N')
    Dif = [np.max(np.abs(R[m] - R['NumPy'])) for m in ['Gauss', 'Gauss-Jordan']]
    Res = [np.max(np.abs(A @ R[m] - b)) for m in ['Gauss', 'Gauss-Jordan', 'NumPy']]
    print(f'{nombre:<12}{len(b):>5}x{len(b):<2}{np.linalg.cond(A):>11.1f}{Dif[0]:>12.1e}{Dif[1]:>11.1e}'
          f'{Res[0]:>12.1e}{Res[1]:>11.1e}{Res[2]:>12.1e}')

Aplicacion    Tamano  Condicion  Dif. Gauss   Dif. G-J  Res. Gauss   Res. G-J  Res. NumPy
Produccion      5x5       236.0     1.2e-14    2.1e-14     2.8e-14    0.0e+00     0.0e+00
Costos          6x6         2.0     2.8e-14    2.8e-14     4.3e-14    1.4e-14     4.3e-14
Mercado         8x8         2.4     1.8e-15    3.6e-15     2.8e-14    2.8e-14     2.8e-14


**Observaciones.**
- En las tres aplicaciones, la eliminación gaussiana y Gauss-Jordan coinciden con `np.linalg.solve` hasta errores de redondeo (diferencias de 10⁻¹⁴ o menores).
- Los residuos |Ax − b| son del orden de 10⁻¹⁴: las soluciones cumplen las ecuaciones originales.
- Las tres matrices están **bien condicionadas** (números de condición pequeños comparados con los de Hilbert), así que nuestras funciones no dan avisos de pivote pequeño y no hay pérdida de exactitud.
- Los sistemas son pequeños, así que los tres métodos tardan unos pocos milisegundos o menos (los tiempos cambian en cada ejecución y dependen de la computadora). La diferencia de velocidad con NumPy solo se nota en sistemas grandes, como el 100×100 de la sección 8.

## 15. Conclusiones y recomendaciones

**Conclusiones**

1. Las funciones del proyecto (eliminación gaussiana y Gauss-Jordan) resolvieron aplicaciones de economía y negocios de **5, 6 y 8 variables** con los mismos resultados que `np.linalg.solve`. Las diferencias fueron de 10⁻¹⁴ o menores.
2. Lo más difícil de cada aplicación es **plantear el sistema**: decidir las variables y de dónde sale cada ecuación. Una vez escrito como Ax = b, resolverlo es una sola línea de código, sin importar el tamaño.
3. La solución matemática no siempre tiene sentido económico. Por eso cada aplicación se verificó con su propio criterio: producciones no negativas, costos que se conservan y oferta igual a demanda.
4. Los escenarios ("¿qué pasa si...?") se analizan cambiando solo el vector b. En el mercado se vio que un choque en el café afecta también los precios del azúcar y la leche.

**Recomendaciones**

- Para sistemas grandes o mal condicionados, usar `np.linalg.solve`, que es más rápido y no depende de una tolerancia fija.
- Si hay más ecuaciones que incógnitas, o restricciones como "producir al menos" o "no gastar más de", el problema ya no es un sistema cuadrado con solución única: se necesitaría Gauss-Jordan (sistemas rectangulares) o programación lineal.
- Los datos son ilustrativos. Para usar estos modelos en una decisión real hay que reemplazarlos por datos de la empresa o del mercado.